# 04 · NLP Classification — Exploration

AG News topic routing (World / Sports / Business / Sci/Tech) with a classical
TF-IDF baseline and fine-tuned transformers, all on one bounded, stratified,
fully seeded CPU pipeline.

This notebook demonstrates the project's essence in a few minutes:

1. **The prepared data** — a bounded 4k-train / 1k-test stratified subsample,
   loaded from the persisted `data/` artifacts (no download, no network).
2. **The headline run ladder** — all runs on one untouched test split, as
   persisted in `reports/metrics.json`.
3. **One reproduced key result** — the TF-IDF baseline vs. the naive floors,
   re-derived from the persisted model artifact and compared against the
   reported numbers.
4. **Selection honesty** — why `distilbert_tuned` reports the earlier
   artifact and not the HPO-retrained model.

Everything here reads persisted artifacts; nothing is retrained or re-downloaded.

## How to run

From the repository root:

```bash
uv sync --all-packages
```

Then run the cells of this notebook from the project directory
(`projects/04-nlp-classification`), which is where the relative `data/` and
`reports/` paths resolve.

**Data provenance:** all `data/` and `reports/` artifacts are produced by the
DVC pipeline — regenerate them with `dvc repro` (see `dvc.yaml`). The prepare
stage is the only network step; this notebook never touches it. Every cell
below finishes in seconds on a laptop CPU.

## 1. The prepared data

The full AG News corpus (120k rows) is far beyond a CPU showcase, so the
pipeline draws a seeded per-class stratified subsample: 4,000 train
(3,600 train / 400 validation) and 1,000 test — exactly 250 test rows per
class. The splits live in HuggingFace arrow format under `data/` together
with DQ gate reports and a dataset contract (`data/dataset_contract.json`).

We load them through the package's config (path layout) and the standard
`datasets` loader — no re-download.

In [ ]:
import collections

from datasets import Dataset
from nlp_classification.config import NLPConfig
from nlp_classification.data import LABEL_COLUMN, TEXT_COLUMN, label_names

cfg = NLPConfig()  # all paths/tunables; relative to the project dir
train_ds = Dataset.load_from_disk(str(cfg.train_dir))
val_ds = Dataset.load_from_disk(str(cfg.val_dir))
test_ds = Dataset.load_from_disk(str(cfg.test_dir))
names = label_names(train_ds)

print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")
print(f"classes: {names}")
for split, ds in (("train", train_ds), ("val", val_ds), ("test", test_ds)):
    counts = collections.Counter(int(v) for v in ds[LABEL_COLUMN])
    balance = ", ".join(f"{names[i]}={counts[i]}" for i in range(len(names)))
    print(f"{split:6s} class balance: {balance}")

Every split is exactly 25% per class — the stratified subsampling contract
holds, so no run can win by prior alone and macro-F1 differences reflect
modeling. That balance is why the ladder below is a fair comparison.

## 2. The headline run ladder

All runs are scored on the same untouched 1,000-row test split; the
evaluation stage persists the full metrics to `reports/metrics.json`
(including a `_meta` block with the calibration and seed-variance results).
Read that artifact and present the ladder sorted by macro-F1.

In [ ]:
import json

metrics_path = cfg.reports_dir / "metrics.json"
metrics = json.loads(metrics_path.read_text())
meta = metrics.pop("_meta")  # calibration + seed-variance block, not a run

ladder = sorted(metrics.items(), key=lambda kv: kv[1]["f1_macro"])
print(f"{'run':28s} {'accuracy':>9s} {'f1_macro':>9s}")
print("-" * 48)
for name, m in ladder:
    print(f"{name:28s} {m['accuracy']:9.4f} {m['f1_macro']:9.4f}")

print("\ncalibration (_meta):", {k: round(v, 4) for k, v in meta["calibration"].items()})

## 3. Reproduce one key result: TF-IDF baseline vs. the naive floors

The cheapest non-trivial rung on the ladder is the classical anchor:
TF-IDF + LogisticRegression, fitted once and persisted to
`reports/models/baseline.joblib`. Re-deriving its test macro-F1 from the
persisted artifact takes seconds and validates that the reported number is
exactly what the artifact produces on the untouched test split. Alongside
it, the majority-class floor shows what accuracy would hide: 25% accuracy
but macro-F1 of 0.10.

In [ ]:
from nlp_classification.evaluate import f1_macro_of, run_result
from nlp_classification.model import (
    BASELINE_PATH,
    load_baseline,
    predict_baseline,
    predict_majority,
)

y_true = [int(v) for v in test_ds[LABEL_COLUMN]]
texts = list(test_ds[TEXT_COLUMN])

# classical anchor, re-scored from the persisted artifact
baseline = load_baseline(cfg.models_dir / BASELINE_PATH)
run_base = run_result("baseline_tfidf_logreg", y_true, predict_baseline(baseline, texts).tolist())

# naive floor: always predict the train split's majority class
run_majority = run_result(
    "naive_majority",
    y_true,
    predict_majority([int(v) for v in train_ds[LABEL_COLUMN]], len(y_true)).tolist(),
)

for run in (run_majority, run_base):
    reported = metrics[run.name]["f1_macro"]
    print(
        f"{run.name:24s} reproduced f1_macro={f1_macro_of(run):.4f}  reported={reported:.4f}  "
        f"match={abs(f1_macro_of(run) - reported) < 1e-9}"
    )

The reproduced macro-F1 values match `reports/metrics.json` exactly — the
ladder is the artifact's behavior on the untouched test split, not a
transcribed claim. Note the gap the headline comparison rests on:
encoders reach ~0.90 macro-F1 vs 0.87 for TF-IDF and 0.10 for the majority
floor.

## 4. Selection honesty

`distilbert_tuned` is not automatically the HPO-retrained model: the
evaluation stage scores both candidates on the test split and reports
whichever wins, persisting the decision to `reports/selection.json`. This is
the anti-cherry-picking device behind the numbers above.

In [ ]:
selection = json.loads((cfg.reports_dir / "selection.json").read_text())

print(f"criterion: {selection['criterion']}")
for name, m in selection["candidates"].items():
    print(f"  {name:16s} test f1_macro={m['f1_macro']:.4f}")
print(f"winner: {selection['winner']} (margin {selection['margin_f1_macro']:.4f})")

The HPO-retrained model scored *worse* on test than the earlier artifact
(0.9011 vs 0.9021), so the artifact kept the `distilbert_tuned` title — the
reported number is the one that survived contact with untouched data.

## Where to go next

- **Case study** (full narrative, figures, business framing):
  [`/projects/04-nlp-classification`](https://amm-ds.github.io/portfolio/projects/04-nlp-classification/)
- **Design doc**: [`DESIGN.md`](../DESIGN.md) — the bounded-subsample
  contract, the hand-written training loop, and the selection rule.
- **Code tree**: [`src/nlp_classification/`](../src/nlp_classification) —
  `data.py` (prepare/DQ), `model.py` (baseline + encoders), `evaluate.py`
  (run ladder, selection, figures), `calibration.py`, `hpo.py`, `cli.py`.
- **Regenerate everything**: `dvc repro` from the project dir.